# OSRM vs Abs_PM vs 经纬度距离 验证实验

## 目标
1. 对同 (Fwy, Dir) 的站点对，计算三种距离
2. 验证 OSRM 双向距离与 Abs_PM 方向的一致性
3. 分析三种距离之间的关系

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
import re
import glob
from datetime import datetime
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============== 配置 ==============
META_DIR = "../d03_meta"
OUTPUT_DIR = "../output/osrm_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# OSRM 服务器 (公共服务器有速率限制，每秒约1次)
OSRM_SERVER = "http://router.project-osrm.org"
# 如果有本地服务器: OSRM_SERVER = "http://localhost:5000"

# 采样参数 (公共服务器限制，只采样部分数据)
MAX_PAIRS_PER_FWY = 10  # 每条 (Fwy, Dir) 最多采样多少对
MAX_TOTAL_PAIRS = 200   # 总共最多查询多少对
RATE_LIMIT_DELAY = 1.1  # 请求间隔（秒）

print("配置完成！")

## 1. 加载元数据

In [ ]:
# 查找最新的元数据文件
meta_files = glob.glob(os.path.join(META_DIR, "d03_text_meta_*.txt"))
if meta_files:
    meta_file = sorted(meta_files)[-1]  # 取最新的
else:
    raise FileNotFoundError("未找到 D3 元数据文件")

print(f"使用元数据: {meta_file}")

META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

meta_df = pd.read_csv(
    meta_file, sep='\t', names=META_COLUMNS, header=0,
    dtype={'ID': str, 'Fwy': str}
)

print(f"总站点数: {len(meta_df)}")
print(f"\n各类型站点数:")
print(meta_df['Type'].value_counts())

In [ ]:
# 坐标有效性检查
# California 大致范围: 32°N-42°N, 114°W-124°W
def is_valid_coord(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False
    if not (32 <= lat <= 42):
        return False
    if not (-124 <= lon <= -114):
        return False
    return True

meta_df['Valid_Coord'] = meta_df.apply(
    lambda r: is_valid_coord(r['Latitude'], r['Longitude']), axis=1
)

print(f"有效坐标站点: {meta_df['Valid_Coord'].sum()} / {len(meta_df)}")

# 只保留有效坐标的站点
meta_valid = meta_df[meta_df['Valid_Coord']].copy()
print(f"\n过滤后站点数: {len(meta_valid)}")

## 2. 距离计算函数

In [ ]:
def calc_distance_pm(pm1, pm2):
    """
    Abs_PM 差值（英里）
    返回有符号值: pm2 - pm1
    """
    return pm2 - pm1


def calc_distance_geo(lat1, lon1, lat2, lon2):
    """
    Haversine 公式计算直线距离（英里）
    """
    R = 3958.8  # 地球半径（英里）
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c


def calc_distance_osrm(lat1, lon1, lat2, lon2, server=OSRM_SERVER, timeout=10):
    """
    OSRM 最短路距离（单向: 点1 → 点2）
    
    返回: {
        'distance_mi': float,  # 英里
        'duration_min': float, # 分钟
        'status': 'ok' | 'no_route' | 'error',
        'error_msg': str
    }
    """
    url = f"{server}/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {
        "overview": "false",
        "steps": "false"
    }
    
    try:
        response = requests.get(url, params=params, timeout=timeout)
        data = response.json()
        
        if data.get("code") != "Ok":
            return {
                'distance_mi': None,
                'duration_min': None,
                'status': 'no_route',
                'error_msg': data.get("code", "Unknown")
            }
        
        route = data["routes"][0]
        return {
            'distance_mi': route["distance"] / 1609.34,  # 米 → 英里
            'duration_min': route["duration"] / 60,       # 秒 → 分钟
            'status': 'ok',
            'error_msg': None
        }
        
    except requests.exceptions.Timeout:
        return {
            'distance_mi': None,
            'duration_min': None,
            'status': 'timeout',
            'error_msg': 'Request timeout'
        }
    except Exception as e:
        return {
            'distance_mi': None,
            'duration_min': None,
            'status': 'error',
            'error_msg': str(e)
        }


def calc_distance_osrm_bidirectional(lat1, lon1, lat2, lon2, server=OSRM_SERVER):
    """
    OSRM 双向查询: 1→2 和 2→1
    """
    result_1to2 = calc_distance_osrm(lat1, lon1, lat2, lon2, server)
    time.sleep(RATE_LIMIT_DELAY)  # 速率限制
    result_2to1 = calc_distance_osrm(lat2, lon2, lat1, lon1, server)
    
    return {
        '1to2': result_1to2,
        '2to1': result_2to1
    }


# 测试函数
print("测试距离计算函数...")

# 取两个站点测试
test_stations = meta_valid.head(2)
s1, s2 = test_stations.iloc[0], test_stations.iloc[1]

print(f"\n站点1: {s1['ID']}, PM={s1['Abs_PM']:.3f}, ({s1['Latitude']:.4f}, {s1['Longitude']:.4f})")
print(f"站点2: {s2['ID']}, PM={s2['Abs_PM']:.3f}, ({s2['Latitude']:.4f}, {s2['Longitude']:.4f})")

dist_pm = calc_distance_pm(s1['Abs_PM'], s2['Abs_PM'])
dist_geo = calc_distance_geo(s1['Latitude'], s1['Longitude'], s2['Latitude'], s2['Longitude'])

print(f"\nAbs_PM 差值: {dist_pm:.3f} mi")
print(f"经纬度直线距离: {dist_geo:.3f} mi")

In [ ]:
# 测试 OSRM (单次)
print("测试 OSRM 连接...")

osrm_result = calc_distance_osrm(
    s1['Latitude'], s1['Longitude'],
    s2['Latitude'], s2['Longitude']
)

print(f"OSRM 结果: {osrm_result}")

if osrm_result['status'] == 'ok':
    print(f"\nOSRM 距离: {osrm_result['distance_mi']:.3f} mi")
    print(f"OSRM 时间: {osrm_result['duration_min']:.1f} min")
    print("\n✓ OSRM 服务器连接正常!")
else:
    print(f"\n✗ OSRM 查询失败: {osrm_result['error_msg']}")

## 3. 构建采样站点对

In [ ]:
def build_sample_pairs(meta_df, max_per_fwy=10, max_total=200):
    """
    构建采样站点对
    
    对每条 (Fwy, Dir):
    - 按 Abs_PM 排序
    - 选取相邻和隔一个的站点对
    - 确保距离有一定范围（近、中、远）
    """
    all_pairs = []
    
    for (fwy, direction), group in meta_df.groupby(['Fwy', 'Dir']):
        sorted_group = group.sort_values('Abs_PM').reset_index(drop=True)
        n = len(sorted_group)
        
        if n < 2:
            continue
        
        pairs_this_fwy = []
        
        # 策略: 选取不同间隔的点对
        # 相邻 (gap=1), 隔1个 (gap=2), 隔2个 (gap=3), 隔4个 (gap=5)
        for gap in [1, 2, 3, 5, 10]:
            for i in range(0, n - gap, max(1, gap)):
                j = i + gap
                if j >= n:
                    break
                
                node_i = sorted_group.iloc[i]
                node_j = sorted_group.iloc[j]
                
                pairs_this_fwy.append({
                    'ID1': node_i['ID'],
                    'ID2': node_j['ID'],
                    'Fwy': fwy,
                    'Dir': direction,
                    'Type1': node_i['Type'],
                    'Type2': node_j['Type'],
                    'PM1': node_i['Abs_PM'],
                    'PM2': node_j['Abs_PM'],
                    'Lat1': node_i['Latitude'],
                    'Lon1': node_i['Longitude'],
                    'Lat2': node_j['Latitude'],
                    'Lon2': node_j['Longitude'],
                    'Gap': gap,
                })
                
                if len(pairs_this_fwy) >= max_per_fwy:
                    break
            
            if len(pairs_this_fwy) >= max_per_fwy:
                break
        
        all_pairs.extend(pairs_this_fwy[:max_per_fwy])
    
    # 打乱并截取
    pairs_df = pd.DataFrame(all_pairs)
    if len(pairs_df) > max_total:
        pairs_df = pairs_df.sample(n=max_total, random_state=42)
    
    return pairs_df.reset_index(drop=True)


# 构建采样对
pairs_df = build_sample_pairs(meta_valid, MAX_PAIRS_PER_FWY, MAX_TOTAL_PAIRS)
print(f"采样站点对数: {len(pairs_df)}")
print(f"覆盖 (Fwy, Dir) 数: {pairs_df.groupby(['Fwy', 'Dir']).ngroups}")

# 预计算 PM 和 Geo 距离
pairs_df['Dist_PM'] = pairs_df.apply(
    lambda r: calc_distance_pm(r['PM1'], r['PM2']), axis=1
)
pairs_df['Dist_Geo'] = pairs_df.apply(
    lambda r: calc_distance_geo(r['Lat1'], r['Lon1'], r['Lat2'], r['Lon2']), axis=1
)

print(f"\nPM 距离范围: {pairs_df['Dist_PM'].min():.2f} ~ {pairs_df['Dist_PM'].max():.2f} mi")
print(f"Geo 距离范围: {pairs_df['Dist_Geo'].min():.2f} ~ {pairs_df['Dist_Geo'].max():.2f} mi")

display(pairs_df.head(10))

## 4. 批量 OSRM 查询

In [ ]:
def batch_osrm_query(pairs_df, server=OSRM_SERVER, rate_limit=1.1):
    """
    批量 OSRM 双向查询
    """
    results = []
    
    for idx, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="OSRM查询"):
        # 1 → 2
        r_1to2 = calc_distance_osrm(
            row['Lat1'], row['Lon1'],
            row['Lat2'], row['Lon2'],
            server
        )
        time.sleep(rate_limit)
        
        # 2 → 1
        r_2to1 = calc_distance_osrm(
            row['Lat2'], row['Lon2'],
            row['Lat1'], row['Lon1'],
            server
        )
        time.sleep(rate_limit)
        
        results.append({
            'idx': idx,
            'OSRM_1to2_mi': r_1to2['distance_mi'],
            'OSRM_2to1_mi': r_2to1['distance_mi'],
            'OSRM_1to2_min': r_1to2['duration_min'],
            'OSRM_2to1_min': r_2to1['duration_min'],
            'Status_1to2': r_1to2['status'],
            'Status_2to1': r_2to1['status'],
        })
    
    return pd.DataFrame(results).set_index('idx')


# 执行查询 (需要一定时间)
print(f"开始 OSRM 查询，预计时间: {len(pairs_df) * 2 * RATE_LIMIT_DELAY / 60:.1f} 分钟")
print(f"(每对需要2次查询，每次间隔 {RATE_LIMIT_DELAY} 秒)")
print("="*60)

osrm_results = batch_osrm_query(pairs_df, OSRM_SERVER, RATE_LIMIT_DELAY)

# 合并结果
pairs_df = pairs_df.join(osrm_results)

print(f"\n查询完成!")
print(f"成功率 (1→2): {(pairs_df['Status_1to2'] == 'ok').mean()*100:.1f}%")
print(f"成功率 (2→1): {(pairs_df['Status_2to1'] == 'ok').mean()*100:.1f}%")

## 5. 方向一致性分析

In [ ]:
# 只分析双向都成功的
valid_pairs = pairs_df[
    (pairs_df['Status_1to2'] == 'ok') & 
    (pairs_df['Status_2to1'] == 'ok')
].copy()

print(f"双向都成功的站点对: {len(valid_pairs)} / {len(pairs_df)}")

In [ ]:
def determine_direction_by_pm(row):
    """
    根据 Abs_PM 和 Dir 判断方向
    
    返回: '1to2' 或 '2to1'，表示车流方向
    
    逻辑:
    - N/E 方向: PM 递增 = 车流方向
      如果 PM2 > PM1，则车流 1→2
    - S/W 方向: PM 递减 = 车流方向
      如果 PM2 < PM1，则车流 1→2
    """
    pm_diff = row['PM2'] - row['PM1']  # 正值 = PM2 更大
    direction = row['Dir']
    
    if direction in ['N', 'E']:
        # N/E: PM 递增方向是车流方向
        return '1to2' if pm_diff > 0 else '2to1'
    else:
        # S/W: PM 递减方向是车流方向
        return '1to2' if pm_diff < 0 else '2to1'


def determine_direction_by_osrm(row, threshold_ratio=0.8):
    """
    根据 OSRM 双向距离判断方向
    
    逻辑:
    - 顺向行驶距离短，逆向需要绕行
    - 如果 1→2 明显短于 2→1，则顺向是 1→2
    
    返回: '1to2', '2to1', 或 'uncertain'
    """
    d_1to2 = row['OSRM_1to2_mi']
    d_2to1 = row['OSRM_2to1_mi']
    
    if pd.isna(d_1to2) or pd.isna(d_2to1):
        return 'unknown'
    
    if d_1to2 < d_2to1 * threshold_ratio:
        return '1to2'
    elif d_2to1 < d_1to2 * threshold_ratio:
        return '2to1'
    else:
        return 'similar'  # 两个方向距离相近


# 计算两种方向判断
valid_pairs['Dir_by_PM'] = valid_pairs.apply(determine_direction_by_pm, axis=1)
valid_pairs['Dir_by_OSRM'] = valid_pairs.apply(determine_direction_by_osrm, axis=1)

# 比较一致性
valid_pairs['Direction_Match'] = valid_pairs['Dir_by_PM'] == valid_pairs['Dir_by_OSRM']

print("方向判断结果分布:")
print(f"\nAbs_PM 方向:")
print(valid_pairs['Dir_by_PM'].value_counts())
print(f"\nOSRM 方向:")
print(valid_pairs['Dir_by_OSRM'].value_counts())

In [ ]:
# 一致性统计
print("="*60)
print("方向判断一致性分析")
print("="*60)

# 排除 OSRM 判断为 similar 的情况
decisive = valid_pairs[valid_pairs['Dir_by_OSRM'].isin(['1to2', '2to1'])]
print(f"\nOSRM 有明确方向判断的: {len(decisive)} / {len(valid_pairs)}")

if len(decisive) > 0:
    match_rate = decisive['Direction_Match'].mean() * 100
    print(f"方向一致率: {match_rate:.1f}%")
    
    # 交叉表
    print(f"\n交叉表:")
    ct = pd.crosstab(decisive['Dir_by_PM'], decisive['Dir_by_OSRM'], margins=True)
    display(ct)

# OSRM similar 的情况
similar = valid_pairs[valid_pairs['Dir_by_OSRM'] == 'similar']
print(f"\nOSRM 双向距离相近 (similar): {len(similar)} ({len(similar)/len(valid_pairs)*100:.1f}%)")
if len(similar) > 0:
    print(f"  这些站点对的 OSRM 距离差异:")
    similar_copy = similar.copy()
    similar_copy['OSRM_Diff'] = (similar_copy['OSRM_1to2_mi'] - similar_copy['OSRM_2to1_mi']).abs()
    similar_copy['OSRM_Ratio'] = similar_copy['OSRM_1to2_mi'] / similar_copy['OSRM_2to1_mi']
    print(f"  差异绝对值: {similar_copy['OSRM_Diff'].describe()[['mean','min','max']]}")
    print(f"  比值: {similar_copy['OSRM_Ratio'].describe()[['mean','min','max']]}")

In [ ]:
# 不一致的案例分析
if len(decisive) > 0:
    mismatch = decisive[~decisive['Direction_Match']]
    print(f"\n方向不一致的案例: {len(mismatch)}")
    
    if len(mismatch) > 0:
        print("\n详细信息:")
        display(mismatch[[
            'ID1', 'ID2', 'Fwy', 'Dir', 'Type1', 'Type2',
            'PM1', 'PM2', 'Dist_PM', 'Dist_Geo',
            'OSRM_1to2_mi', 'OSRM_2to1_mi',
            'Dir_by_PM', 'Dir_by_OSRM'
        ]])

## 6. 三种距离的关系分析

In [ ]:
# 计算 OSRM 顺向距离（取较短的那个方向）
valid_pairs['OSRM_Forward_mi'] = valid_pairs[['OSRM_1to2_mi', 'OSRM_2to1_mi']].min(axis=1)
valid_pairs['OSRM_Backward_mi'] = valid_pairs[['OSRM_1to2_mi', 'OSRM_2to1_mi']].max(axis=1)

# 计算各种比值
valid_pairs['Abs_Dist_PM'] = valid_pairs['Dist_PM'].abs()
valid_pairs['Ratio_OSRM_to_PM'] = valid_pairs['OSRM_Forward_mi'] / valid_pairs['Abs_Dist_PM']
valid_pairs['Ratio_OSRM_to_Geo'] = valid_pairs['OSRM_Forward_mi'] / valid_pairs['Dist_Geo']
valid_pairs['Ratio_PM_to_Geo'] = valid_pairs['Abs_Dist_PM'] / valid_pairs['Dist_Geo']
valid_pairs['Ratio_Backward_to_Forward'] = valid_pairs['OSRM_Backward_mi'] / valid_pairs['OSRM_Forward_mi']

print("三种距离的关系统计:")
print("="*60)

print(f"\n1. OSRM顺向 / Abs_PM:")
print(valid_pairs['Ratio_OSRM_to_PM'].describe())

print(f"\n2. OSRM顺向 / 经纬度直线距离:")
print(valid_pairs['Ratio_OSRM_to_Geo'].describe())

print(f"\n3. Abs_PM / 经纬度直线距离:")
print(valid_pairs['Ratio_PM_to_Geo'].describe())

print(f"\n4. OSRM逆向 / OSRM顺向:")
print(valid_pairs['Ratio_Backward_to_Forward'].describe())

In [ ]:
# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. OSRM顺向 vs Abs_PM
ax = axes[0, 0]
ax.scatter(valid_pairs['Abs_Dist_PM'], valid_pairs['OSRM_Forward_mi'], alpha=0.5, s=20)
max_val = max(valid_pairs['Abs_Dist_PM'].max(), valid_pairs['OSRM_Forward_mi'].max())
ax.plot([0, max_val], [0, max_val], 'r--', label='y=x')
ax.set_xlabel('Abs_PM Distance (mi)')
ax.set_ylabel('OSRM Forward Distance (mi)')
ax.set_title('OSRM Forward vs Abs_PM')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. OSRM顺向 vs 经纬度
ax = axes[0, 1]
ax.scatter(valid_pairs['Dist_Geo'], valid_pairs['OSRM_Forward_mi'], alpha=0.5, s=20)
max_val = max(valid_pairs['Dist_Geo'].max(), valid_pairs['OSRM_Forward_mi'].max())
ax.plot([0, max_val], [0, max_val], 'r--', label='y=x')
ax.set_xlabel('Haversine Distance (mi)')
ax.set_ylabel('OSRM Forward Distance (mi)')
ax.set_title('OSRM Forward vs Haversine')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Abs_PM vs 经纬度
ax = axes[0, 2]
ax.scatter(valid_pairs['Dist_Geo'], valid_pairs['Abs_Dist_PM'], alpha=0.5, s=20)
max_val = max(valid_pairs['Dist_Geo'].max(), valid_pairs['Abs_Dist_PM'].max())
ax.plot([0, max_val], [0, max_val], 'r--', label='y=x')
ax.set_xlabel('Haversine Distance (mi)')
ax.set_ylabel('Abs_PM Distance (mi)')
ax.set_title('Abs_PM vs Haversine')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. OSRM 1to2 vs 2to1
ax = axes[1, 0]
ax.scatter(valid_pairs['OSRM_1to2_mi'], valid_pairs['OSRM_2to1_mi'], alpha=0.5, s=20)
max_val = max(valid_pairs['OSRM_1to2_mi'].max(), valid_pairs['OSRM_2to1_mi'].max())
ax.plot([0, max_val], [0, max_val], 'r--', label='y=x')
ax.set_xlabel('OSRM 1→2 (mi)')
ax.set_ylabel('OSRM 2→1 (mi)')
ax.set_title('OSRM Bidirectional')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. 比值分布: OSRM/PM
ax = axes[1, 1]
valid_ratio = valid_pairs['Ratio_OSRM_to_PM'].replace([np.inf, -np.inf], np.nan).dropna()
ax.hist(valid_ratio[valid_ratio < 5], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=1, color='red', linestyle='--', label='Ratio=1')
ax.set_xlabel('OSRM / Abs_PM Ratio')
ax.set_ylabel('Count')
ax.set_title('OSRM / Abs_PM Ratio Distribution')
ax.legend()

# 6. 比值分布: Backward/Forward
ax = axes[1, 2]
valid_ratio = valid_pairs['Ratio_Backward_to_Forward'].replace([np.inf, -np.inf], np.nan).dropna()
ax.hist(valid_ratio[valid_ratio < 10], bins=50, edgecolor='black', alpha=0.7, color='green')
ax.axvline(x=1, color='red', linestyle='--', label='Ratio=1')
ax.set_xlabel('OSRM Backward / Forward Ratio')
ax.set_ylabel('Count')
ax.set_title('OSRM Direction Asymmetry')
ax.legend()

plt.suptitle('Distance Metrics Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'distance_comparison.png'), dpi=150)
plt.show()

In [ ]:
# 异常值检测
print("="*60)
print("异常值检测")
print("="*60)

# 1. OSRM/PM 比值异常 (偏离 1.0 太多)
ratio_anomaly = valid_pairs[
    (valid_pairs['Ratio_OSRM_to_PM'] < 0.5) | 
    (valid_pairs['Ratio_OSRM_to_PM'] > 2.0)
]
print(f"\n1. OSRM/PM 比值异常 (<0.5 或 >2.0): {len(ratio_anomaly)}")
if len(ratio_anomaly) > 0:
    display(ratio_anomaly[[
        'ID1', 'ID2', 'Fwy', 'Dir', 'Type1', 'Type2',
        'Abs_Dist_PM', 'Dist_Geo', 'OSRM_Forward_mi',
        'Ratio_OSRM_to_PM'
    ]].head(10))

# 2. 双向距离差异大 (可能有单向路)
direction_anomaly = valid_pairs[valid_pairs['Ratio_Backward_to_Forward'] > 3]
print(f"\n2. OSRM 双向距离差异大 (逆/顺 > 3): {len(direction_anomaly)}")
if len(direction_anomaly) > 0:
    display(direction_anomaly[[
        'ID1', 'ID2', 'Fwy', 'Dir', 'Type1', 'Type2',
        'OSRM_1to2_mi', 'OSRM_2to1_mi', 'Ratio_Backward_to_Forward'
    ]].head(10))

## 7. 保存结果

In [ ]:
# 保存完整结果
output_file = os.path.join(OUTPUT_DIR, 'osrm_validation_results.csv')
valid_pairs.to_csv(output_file, index=False)
print(f"结果已保存: {output_file}")

# 汇总报告
report = f"""
OSRM vs Abs_PM vs 经纬度距离 验证报告
{'='*60}

数据概况:
  元数据文件: {meta_file}
  总站点数: {len(meta_df)}
  采样站点对: {len(pairs_df)}
  OSRM双向成功: {len(valid_pairs)}

方向判断一致性:
  OSRM有明确方向: {len(decisive) if 'decisive' in dir() else 'N/A'}
  PM与OSRM一致率: {match_rate:.1f}% (如果OSRM有明确方向)
  OSRM双向相近: {len(similar) if 'similar' in dir() else 'N/A'}

距离比值统计:
  OSRM顺向/PM:  均值={valid_pairs['Ratio_OSRM_to_PM'].mean():.2f}, 中位数={valid_pairs['Ratio_OSRM_to_PM'].median():.2f}
  OSRM顺向/Geo: 均值={valid_pairs['Ratio_OSRM_to_Geo'].mean():.2f}, 中位数={valid_pairs['Ratio_OSRM_to_Geo'].median():.2f}
  PM/Geo:       均值={valid_pairs['Ratio_PM_to_Geo'].mean():.2f}, 中位数={valid_pairs['Ratio_PM_to_Geo'].median():.2f}
  OSRM逆/顺:    均值={valid_pairs['Ratio_Backward_to_Forward'].mean():.2f}, 中位数={valid_pairs['Ratio_Backward_to_Forward'].median():.2f}

结论:
  1. OSRM距离与Abs_PM距离是否一致: (根据比值判断)
  2. OSRM能否判断方向: (根据双向距离差异判断)
  3. 异常情况: (列出)
"""

report_file = os.path.join(OUTPUT_DIR, 'osrm_validation_report.txt')
with open(report_file, 'w') as f:
    f.write(report)
print(f"报告已保存: {report_file}")

## 8. 结论

运行完成后，根据结果回答以下问题:

1. **OSRM 距离与 Abs_PM 距离的关系?**
   - 比值是否接近 1.0?
   - 有哪些异常情况?

2. **OSRM 能否判断方向?**
   - 双向距离差异有多大?
   - 与 Abs_PM 判断的方向是否一致?

3. **三种距离的可靠性排序?**
   - 哪种最适合做初筛?
   - 哪种最适合做方向判断?